# Completing Router — Test Constraints (Colab)

Tests the classical `RipUpRerouteRouter` (Stage 1+2 of the completing-router overhaul): plain
multi-net completion, **differential-pair coupling**, **length-matched tuning**, trace **width**,
and multi-layer **via escape** — with a board visualization at the end.

This is pure classical routing (A* + rip-up-and-reroute); no ML/training involved. Safe and fast
to run here even on CPU.

In [ ]:
import os, sys
REPO = '/content/Router'
if not os.path.exists(REPO):
    !git clone https://github.com/Klutzhehe/Router.git $REPO
os.chdir(REPO)
!git pull --ff-only
if REPO not in sys.path:
    sys.path.insert(0, REPO)
!pip -q install scipy pyyaml
print('ready')

## 1. Generate a board with diff pairs + length matching + multi-layer

In [ ]:
from pcb_router.data.board_generator import BoardGenerator, BoardConfig

config = BoardConfig()
config.board_width = 220
config.board_height = 220
config.num_nets = 8
config.num_layers = 2
config.num_components = 6
config.diff_pairs = True
config.num_diff_pairs = 1
config.length_matching = True
config.matched_group_size = 2
config.seed = 42

board = BoardGenerator().generate(config)
diff_nets = [n for n in board.nets if n.is_diff_pair]
matched_nets = [n for n in board.nets if n.matched_group_id is not None]
print(f'board {board.width}x{board.height}  layers={board.num_layers}  nets={len(board.nets)}')
print('diff-pair nets   :', [n.name for n in diff_nets])
print('length-matched   :', [(n.name, n.target_length, n.length_tolerance) for n in matched_nets])

## 2. Route it

In [ ]:
import time
from pcb_router.routing.rip_up_router import RipUpRerouteRouter

router = RipUpRerouteRouter(board, max_iterations=20)
t0 = time.time()
res = router.route()
print(f'routed in {time.time()-t0:.1f}s')
print(f"completed: {res['completed']}/{res['total']} ({res['completion_rate']:.0%})  "
      f"iterations={res['iterations']}  converged={res['converged']}  "
      f"vias={len(res['board_state'].vias)}  shared_cells={res['shared_cells']}")

## 3. Check DRC

In [ ]:
from pcb_router.env.drc_checker import DRCChecker

drc = DRCChecker(board.design_rules, router.resolution)
violations = drc.check_all(res['board_state'], res['board_state'].traces, res['board_state'].vias, board)
errors = [v for v in violations if v.severity == 'error' and v.type != 'unconnected']
print(f'total violations: {len(violations)}  |  hard errors (excl. unconnected): {len(errors)}')
for v in violations[:20]:
    print(f'  [{v.type}] {v.severity}: {v.description}  (x={v.x}, y={v.y}, layer={v.layer})')

## 4. Verify differential-pair coupling

P and N traces should stay a roughly constant distance apart along their length.

In [ ]:
import numpy as np

if len(diff_nets) >= 2:
    p_net = next(n for n in diff_nets if 'DIFF_P' in n.name or '_P' in n.name)
    n_net = next(n for n in diff_nets if 'DIFF_N' in n.name or '_N' in n.name)
    path_p, path_n = res['routes'][p_net.id], res['routes'][n_net.id]
    if path_p and path_n:
        n_pts = min(len(path_p), len(path_n))
        gaps = [np.hypot(path_p[i][0]-path_n[i][0], path_p[i][1]-path_n[i][1]) * router.resolution
                for i in range(1, n_pts - 1)]
        print(f'{p_net.name}/{n_net.name}: mean gap {np.mean(gaps):.3f} mm, std {np.std(gaps):.3f} mm '
              f'(std should be small -> consistent coupling)')
    else:
        print('diff pair did not fully route:', bool(path_p), bool(path_n))
else:
    print('no diff pair on this board')

## 5. Verify length-matched tuning

Actual trace length vs. each net's `target_length`, within `length_tolerance`.

In [ ]:
for net in matched_nets:
    segs = [t for t in res['board_state'].traces if t.net_id == net.id]
    length = sum(np.hypot((s.end_x - s.start_x) * router.resolution,
                          (s.end_y - s.start_y) * router.resolution) for s in segs)
    err = abs(length - net.target_length)
    ok = err <= net.length_tolerance
    print(f"{net.name}: target={net.target_length:.2f}mm  actual={length:.2f}mm  "
          f"error={err:.2f}mm  tol={net.length_tolerance:.2f}mm  {'OK' if ok else 'OUT OF TOLERANCE'}")

## 6. Visualize the routed board

Diff-pair pins get a magenta ring; length-matched pins get a gold ring. Trace color = net id.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as patches

bs = res['board_state']
diff_ids = {n.id for n in diff_nets}
matched_ids = {n.id for n in matched_nets}
net_ids = sorted({n.id for n in board.nets if n.id != 0})
cmap = plt.get_cmap('tab10')
net_color = {nid: cmap(i % 10) for i, nid in enumerate(net_ids)}

fig, ax = plt.subplots(figsize=(8, 8))
ax.set_facecolor('#111222')
fig.patch.set_facecolor('#111222')
ax.set_xlim(0, board.width); ax.set_ylim(0, board.height); ax.set_aspect('equal')

for comp in board.components:
    ax.add_patch(patches.Rectangle((comp.x, comp.y), comp.width, comp.height,
                                    facecolor='#1E2035', edgecolor='#374151', lw=1.2, alpha=0.85))

for seg in bs.traces:
    col = net_color.get(seg.net_id, '#888888')
    lw = max(1.2, seg.width / router.resolution)
    ax.plot([seg.start_x, seg.end_x], [seg.start_y, seg.end_y], color=col, lw=lw,
            alpha=0.95, solid_capstyle='round')

for via in bs.vias:
    r = (via.drill_size / 2 + via.annular_ring) / router.resolution
    ax.add_patch(patches.Circle((via.x, via.y), r, fc='#EAB308', ec='white', lw=0.6, alpha=0.9))

for pin in board.pins.values():
    col = net_color.get(pin.net_id, '#6B7280')
    ring = None
    if pin.net_id in diff_ids:
        ring = '#EC4899'   # magenta = diff pair
    elif pin.net_id in matched_ids:
        ring = '#EAB308'   # gold = length matched
    ax.add_patch(patches.Circle((pin.global_x, pin.global_y), 3, fc=col,
                                ec=(ring or 'white'), lw=(2.2 if ring else 0.6), alpha=0.95, zorder=5))

ax.set_xticks([]); ax.set_yticks([])
ax.set_title(f"completed {res['completed']}/{res['total']}  |  magenta ring=diff-pair  gold ring=length-matched",
             color='#E2E8F0', fontsize=10)
plt.tight_layout()
plt.show()